# Système expert et chaînage avant

- **Cours** : Intelligence artificielle · Master 2 (ANG, LGC, DID) · Université Yahia Farès, Médéa
- **Chapitre 3** : Types d'IA et approches · IA symbolique : les systèmes experts
- **Durée** : environ 1 h 30

### Ce que vous allez faire
- Lire la base de règles du cours, qui reconnaît les formes poétiques (haïku, sonnet italien ou français, sonnet anglais), et décrire un poème par une base de faits.
- Faire raisonner un moteur d'inférence en chaînage avant, cycle par cycle, puis en chaînage arrière.
- Dérouler l'exercice 1 du TD du chapitre 3 pour vérifier votre corrigé, puis éprouver les limites d'un système expert.

### Mode d'emploi
Exécutez chaque cellule de haut en bas : cliquez sur le bouton ▶ à sa gauche, ou appuyez sur Maj + Entrée. **Vous n'avez jamais à écrire de code.** Modifiez seulement les champs des cellules intitulées « À MODIFIER ».

- Dans Google Colab, le menu « Exécution → Tout exécuter » (*Runtime → Run all*) lance tout le carnet ; les champs « À MODIFIER » s'affichent comme des cases à cocher, des zones de texte ou des listes.
- Dans Jupyter, ces champs sont de simples commentaires `#@param` : changez seulement la valeur placée après le signe `=` (`True` pour une case cochée, `False` sinon).
- Après avoir modifié une valeur, relancez la cellule, puis les cellules qui la suivent.

Avant chaque cellule, **prédisez** le résultat ; après, comparez avec la rubrique « Ce que vous devez voir ».

## 1. Les trois composants d'un système expert
- La **base de faits** : ce que l'on sait du poème étudié (« le poème a 14 vers »). Elle s'enrichit au fil du raisonnement.
- La **base de règles** : le savoir du domaine, sous la forme SI prémisses ALORS conclusion. Les prémisses sont reliées par ET.
- Le **moteur d'inférence** : le programme qui repère les règles applicables, les déclenche et ajoute leurs conclusions à la base de faits.

Voici la base de règles du cours, dans l'ordre de priorité :

| Règle | SI (prémisses) | ALORS (conclusion) |
|---|---|---|
| R1 | 3 vers ET syllabes 5-7-5 | haïku |
| R2 | 14 vers ET 2 quatrains ET 2 tercets | sonnet |
| R3 | 14 vers ET 3 quatrains ET 1 distique | sonnet |
| R4 | sonnet ET 2 tercets | sonnet italien ou français |
| R5 | sonnet ET 1 distique | sonnet anglais |
| R6 | sonnet | forme fixe |
| R7 | haïku | forme fixe |

En Python, chaque règle est une paire (prémisses, conclusion). Les prémisses forment un **ensemble**, écrit entre accolades : l'ordre n'y compte pas. `enumerate(regles, start=1)` numérote les règles à partir de 1, et `" ET ".join(...)` assemble les prémisses avec « ET ».

**Avant d'exécuter :** combien de lignes la cellule va-t-elle afficher ?

In [ ]:
#@title La base de règles du cours, dans l'ordre de priorité
# Une règle = (ensemble des prémisses, conclusion)
regles = [
    ({"3 vers", "syllabes 5-7-5"}, "haïku"),                     # R1
    ({"14 vers", "2 quatrains", "2 tercets"}, "sonnet"),         # R2
    ({"14 vers", "3 quatrains", "1 distique"}, "sonnet"),        # R3
    ({"sonnet", "2 tercets"}, "sonnet italien ou français"),     # R4
    ({"sonnet", "1 distique"}, "sonnet anglais"),                # R5
    ({"sonnet"}, "forme fixe"),                                  # R6
    ({"haïku"}, "forme fixe"),                                   # R7
]
for numero, (premisses, conclusion) in enumerate(regles, start=1):
    print(f"R{numero} : SI {' ET '.join(sorted(premisses))} ALORS {conclusion}")

### Ce que vous devez voir
```
R1 : SI 3 vers ET syllabes 5-7-5 ALORS haïku
R2 : SI 14 vers ET 2 quatrains ET 2 tercets ALORS sonnet
R3 : SI 1 distique ET 14 vers ET 3 quatrains ALORS sonnet
R4 : SI 2 tercets ET sonnet ALORS sonnet italien ou français
R5 : SI 1 distique ET sonnet ALORS sonnet anglais
R6 : SI sonnet ALORS forme fixe
R7 : SI haïku ALORS forme fixe
```
Sept lignes, une par règle. Les prémisses sont affichées dans l'ordre alphabétique (`sorted`), où les chiffres passent avant les lettres : l'ordre des prémisses ne change rien à une règle.

**Observez :** deux règles concluent « sonnet » et deux concluent « forme fixe ». Il suffit que l'une ou l'autre se déclenche : les règles qui aboutissent au même fait sont reliées par OU.

## 2. Les faits non déductibles
Un fait **non déductible** n'apparaît dans la conclusion d'aucune règle : seuls l'observation ou l'utilisateur peuvent le fournir. Pour les trouver, on réunit toutes les prémisses, puis on retire toutes les conclusions. Sur des ensembles, `|` réunit et `-` retire.

**Avant d'exécuter :** citez deux faits non déductibles de cette base de règles.

In [ ]:
#@title Quels faits sont non déductibles ?
toutes_les_premisses = set()
toutes_les_conclusions = set()
for premisses, conclusion in regles:
    toutes_les_premisses = toutes_les_premisses | premisses   # | : réunir deux ensembles
    toutes_les_conclusions.add(conclusion)
non_deductibles = toutes_les_premisses - toutes_les_conclusions   # - : retirer
print("Faits non déductibles :", sorted(non_deductibles))
print("Faits déductibles :", sorted(toutes_les_conclusions))

### Ce que vous devez voir
```
Faits non déductibles : ['1 distique', '14 vers', '2 quatrains', '2 tercets', '3 quatrains', '3 vers', 'syllabes 5-7-5']
Faits déductibles : ['forme fixe', 'haïku', 'sonnet', 'sonnet anglais', 'sonnet italien ou français']
```
Les faits non déductibles sont des **observations** : on compte les vers, les strophes, les syllabes. « sonnet » est à la fois prémisse (de R4, R5, R6) et conclusion (de R2, R3) : c'est un fait intermédiaire.

**Observez :** qui fournit les observations dans un vrai système expert ? Quelle compétence de lecteur faut-il pour cocher « 2 tercets » ?

## 3. Le moteur d'inférence : le chaînage avant
Le **chaînage avant** part des faits. À chaque **cycle**, le moteur parcourt les règles dans l'ordre de priorité, déclenche la **première** règle applicable, ajoute sa conclusion à la base de faits, puis repart de R1 : c'est la **résolution de conflit** du cours. Une règle est applicable quand toutes ses prémisses sont connues et que sa conclusion est nouvelle, ce qui empêche une règle de se déclencher deux fois. Le moteur s'arrête quand aucune règle ne produit de fait nouveau.

`premisses <= faits` signifie « l'ensemble des prémisses est inclus dans la base de faits ». `break` interrompt le parcours des règles pour repartir de R1.

**Avant d'exécuter :** cette cellule définit une **fonction** avec `def`. Que va-t-elle afficher ?

In [ ]:
#@title Le moteur d'inférence en chaînage avant
def chainage_avant(faits, regles):
    faits = set(faits)                      # copie : la base de départ ne change pas
    cycle = 1
    while True:
        regle_declenchee = False
        for numero, (premisses, conclusion) in enumerate(regles, start=1):
            # règle applicable : prémisses toutes connues, conclusion nouvelle
            if premisses <= faits and conclusion not in faits:
                faits.add(conclusion)
                print(f"Cycle {cycle} : R{numero} déclenchée → {conclusion}")
                regle_declenchee = True
                break                       # résolution de conflit : on repart de R1
        if not regle_declenchee:
            print(f"Cycle {cycle} : aucune règle ne produit de fait nouveau → arrêt")
            return faits
        cycle = cycle + 1

### Ce que vous devez voir
Rien ! Une fonction est une recette : `def` l'écrit, mais ne la fait pas encore. Elle servira dans les cellules suivantes, sous le nom `chainage_avant`. Ce moteur ne sait rien de la poésie : on pourrait lui donner des règles de médecine ou de grammaire sans changer une ligne.

## 4. Décrire un poème
Cochez les observations qui décrivent le poème. Par défaut, les cases décrivent le sonnet de Joachim Du Bellay « Heureux qui, comme Ulysse, a fait un beau voyage » (*Les Regrets*, 1558) : 14 vers, 2 quatrains, 2 tercets, en alexandrins. La cellule range ces observations dans un dictionnaire : chaque fait (la clé) reçoit `True` si la case est cochée.

**Avant d'exécuter :** combien de valeurs `True` le dictionnaire contiendra-t-il ?

In [ ]:
#@title À MODIFIER : décrivez le poème (par défaut, le sonnet de Du Bellay)
quatorze_vers = True  #@param {type:"boolean"}
trois_vers = False  #@param {type:"boolean"}
deux_quatrains = True  #@param {type:"boolean"}
trois_quatrains = False  #@param {type:"boolean"}
deux_tercets = True  #@param {type:"boolean"}
un_distique = False  #@param {type:"boolean"}
syllabes_5_7_5 = False  #@param {type:"boolean"}
alexandrins = True  #@param {type:"boolean"}

observations = {"14 vers": quatorze_vers, "3 vers": trois_vers, "2 quatrains": deux_quatrains,
                "3 quatrains": trois_quatrains, "2 tercets": deux_tercets,
                "1 distique": un_distique, "syllabes 5-7-5": syllabes_5_7_5, "alexandrins": alexandrins}
print(observations)

### Ce que vous devez voir
```
{'14 vers': True, '3 vers': False, '2 quatrains': True, '3 quatrains': False, '2 tercets': True, '1 distique': False, 'syllabes 5-7-5': False, 'alexandrins': True}
```
Quatre cases cochées, donc quatre valeurs `True`.

**Avant d'exécuter la cellule suivante :** avec la base de faits {14 vers, 2 quatrains, 2 tercets, alexandrins}, quelles règles vont se déclencher, dans quel ordre, et en combien de cycles ? Faites le raisonnement sur papier avec le tableau des règles.

In [ ]:
#@title Lancer le chaînage avant sur ce poème
base = set()
for fait, observe in observations.items():
    if observe:                                 # on garde seulement les cases cochées
        base.add(fait)
print("Base de faits initiale :", sorted(base))
resultat = chainage_avant(base, regles)
print("Faits déduits :", sorted(resultat - base))

### Ce que vous devez voir
```
Base de faits initiale : ['14 vers', '2 quatrains', '2 tercets', 'alexandrins']
Cycle 1 : R2 déclenchée → sonnet
Cycle 2 : R4 déclenchée → sonnet italien ou français
Cycle 3 : R6 déclenchée → forme fixe
Cycle 4 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['forme fixe', 'sonnet', 'sonnet italien ou français']
```
C'est le déroulement du cours : au cycle 1, R1 échoue et R2 s'applique ; au cycle 2, le moteur repart de R1, R2 est déjà faite (sa conclusion est connue) et R4 s'applique ; au cycle 3, R5 échoue et R6 s'applique ; au cycle 4, rien de nouveau. Trois faits ont été déduits. Aucune règle n'utilise « alexandrins » : ce fait reste inerte.

**Modifiez :** décrivez le sonnet 18 de Shakespeare, « Shall I compare thee to a summer's day? » (1609) : décochez `deux_quatrains`, `deux_tercets` et `alexandrins`, cochez `trois_quatrains` et `un_distique`. Relancez les deux cellules. Quelles règles se déclenchent maintenant ?

## 5. Plusieurs poèmes, et un poème que le système ne reconnaît pas
La cellule lance le même moteur sur quatre descriptions : le sonnet de Du Bellay, le sonnet 18 de Shakespeare, le haïku de Bashō « Le vieil étang » (1686, 5-7-5 syllabes en japonais) et un sonnet imprimé d'un seul bloc, sans blancs entre les strophes : on n'y observe que 14 vers.

**Avant d'exécuter :** le sonnet imprimé d'un seul bloc sera-t-il reconnu comme un sonnet ?

In [ ]:
#@title Le même moteur sur quatre poèmes
poemes = {
    "Du Bellay, « Heureux qui, comme Ulysse... » (1558)": {"14 vers", "2 quatrains", "2 tercets", "alexandrins"},
    "Shakespeare, sonnet 18 (1609)": {"14 vers", "3 quatrains", "1 distique"},
    "Bashō, « Le vieil étang » (1686)": {"3 vers", "syllabes 5-7-5"},
    "Un sonnet imprimé d'un seul bloc": {"14 vers"},
}
for titre, faits in poemes.items():
    print("---", titre)
    resultat = chainage_avant(faits, regles)
    print("Faits déduits :", sorted(resultat - faits))

### Ce que vous devez voir
```
--- Du Bellay, « Heureux qui, comme Ulysse... » (1558)
Cycle 1 : R2 déclenchée → sonnet
Cycle 2 : R4 déclenchée → sonnet italien ou français
Cycle 3 : R6 déclenchée → forme fixe
Cycle 4 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['forme fixe', 'sonnet', 'sonnet italien ou français']
--- Shakespeare, sonnet 18 (1609)
Cycle 1 : R3 déclenchée → sonnet
Cycle 2 : R5 déclenchée → sonnet anglais
Cycle 3 : R6 déclenchée → forme fixe
Cycle 4 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['forme fixe', 'sonnet', 'sonnet anglais']
--- Bashō, « Le vieil étang » (1686)
Cycle 1 : R1 déclenchée → haïku
Cycle 2 : R7 déclenchée → forme fixe
Cycle 3 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['forme fixe', 'haïku']
--- Un sonnet imprimé d'un seul bloc
Cycle 1 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : []
```
Le sonnet anglais passe par R3, R5 et R6, le haïku par R1 et R7. Le sonnet imprimé d'un seul bloc ne déclenche **aucune** règle : là où un lecteur reconnaît aussitôt la forme, le système échoue dès qu'on sort des cas prévus. C'est la **fragilité** des systèmes experts, l'une des causes de leur déclin à la fin des années 1980.

**Observez :** en revanche, chaque conclusion s'explique par la liste exacte des règles déclenchées. Quel avantage cette **explicabilité** offre-t-elle à un enseignant qui corrige un commentaire de texte ?

## 6. Le chaînage arrière : partir du but
Le **chaînage arrière** part de l'hypothèse à prouver. Il cherche les règles qui la concluent ; leurs prémisses deviennent des **sous-buts**, jusqu'à retomber sur des faits de la base. Un sous-but non déductible et absent de la base fait échouer la piste.

La fonction `prouver` s'appelle elle-même pour chaque sous-but, avec une marge plus grande pour que la trace se lise comme un plan. `all(...)` vaut vrai si toutes les prémisses sont établies, et s'arrête dès la première qui échoue. La base de faits est celle de l'étape 4 (par défaut, Du Bellay).

**Avant d'exécuter :** l'hypothèse « sonnet anglais » peut-elle être établie pour le sonnet de Du Bellay ? Quelle prémisse manque ?

In [ ]:
#@title À MODIFIER : tester une hypothèse en chaînage arrière
hypothese = "sonnet anglais"  #@param ["sonnet anglais", "sonnet italien ou français", "forme fixe", "haïku"]

def prouver(but, faits, regles, marge=""):
    if but in faits:                                   # le but est déjà un fait connu
        print(marge + f"« {but} » est dans la base de faits : établi")
        return True
    for numero, (premisses, conclusion) in enumerate(regles, start=1):
        if conclusion == but:                          # une règle qui conclut le but
            print(marge + f"Pour établir « {but} », R{numero} demande : {sorted(premisses)}")
            if all(prouver(p, faits, regles, marge + "    ") for p in sorted(premisses)):
                return True                            # tous les sous-buts sont établis
    print(marge + f"« {but} » ne peut pas être établi")
    return False

print("Hypothèse établie :", prouver(hypothese, base, regles))

### Ce que vous devez voir
```
Pour établir « sonnet anglais », R5 demande : ['1 distique', 'sonnet']
    « 1 distique » ne peut pas être établi
« sonnet anglais » ne peut pas être établi
Hypothèse établie : False
```
Par R5, « sonnet anglais » exige « 1 distique » et « sonnet ». « 1 distique » est non déductible et absent de la base : l'hypothèse est rejetée aussitôt, sans même examiner « sonnet ».

**Modifiez :** choisissez `forme fixe` dans la liste. La trace descend de « forme fixe » à « sonnet » (R6), puis de « sonnet » aux trois observations (R2). Comparez avec le chaînage avant de l'étape 4 : lequel des deux examine le moins de règles ? Lequel convient au diagnostic, quand on a une hypothèse à vérifier ?

## 7. TD du chapitre 3, exercice 1
Soit les règles suivantes en chaînage avant. Le symbole ∧ signifie « ET » (le programme du cours l'écrit ∩) : « E ∧ G → C » se lit « si E et G sont établis, alors C est établi ».

| Règle | Conditions | Conclusions |
|---|---|---|
| R1 | A | E |
| R2 | B | D |
| R3 | H | A ∧ F |
| R4 | E ∧ G | C |
| R5 | E ∧ K | B |
| R6 | D ∧ E ∧ K | C |
| R7 | G ∧ K ∧ F | A |

- **Ordre des règles** : R3 R7 R1 R4 R5 R2 R6
- **Faits non déductibles** : H, G, K
- **Base de faits initiale** : H, K
- **But** : C

Questions : a) faire le graphe correspondant aux règles ; b) dérouler le système en chaînage avant. Cherchez d'abord sur papier, puis vérifiez avec les cellules suivantes.

Une règle peut ici avoir plusieurs conclusions (R3 conclut A et F) : chaque règle est rangée dans un dictionnaire, sous son nom, avec l'ensemble de ses conditions et l'ensemble de ses conclusions.

**Avant d'exécuter :** quelles règles concluent A ? Lesquelles concluent C ?

In [ ]:
#@title Les règles de l'exercice 1 du TD
# nom de la règle : (ensemble des conditions, ensemble des conclusions)
regles_td = {
    "R1": ({"A"}, {"E"}),
    "R2": ({"B"}, {"D"}),
    "R3": ({"H"}, {"A", "F"}),
    "R4": ({"E", "G"}, {"C"}),
    "R5": ({"E", "K"}, {"B"}),
    "R6": ({"D", "E", "K"}, {"C"}),
    "R7": ({"G", "K", "F"}, {"A"}),
}
for nom, (conditions, conclusions) in regles_td.items():
    print(f"{nom} : {' ∧ '.join(sorted(conditions))} → {' ∧ '.join(sorted(conclusions))}")

### Ce que vous devez voir
```
R1 : A → E
R2 : B → D
R3 : H → A ∧ F
R4 : E ∧ G → C
R5 : E ∧ K → B
R6 : D ∧ E ∧ K → C
R7 : F ∧ G ∧ K → A
```
Les conditions sont affichées dans l'ordre alphabétique : R7 s'écrit ici « F ∧ G ∧ K », ce qui revient au même que « G ∧ K ∧ F ». A est conclu par R3 et R7, C par R4 et R6.

### a) Le graphe des règles
Dans le graphe ET/OU, chaque fait est un rectangle et chaque règle un cercle. Une flèche va de chaque condition vers la règle, puis de la règle vers chacune de ses conclusions. La bibliothèque `networkx` construit ce graphe orienté ; `in_degree(n)` compte les flèches qui arrivent sur `n`.

**Avant d'exécuter :** combien de flèches le graphe compte-t-il ? Sur quels faits n'arrive aucune flèche ?

In [ ]:
#@title a) Construire le graphe ET/OU des règles
import networkx as nx

graphe = nx.DiGraph()                        # un graphe orienté : des sommets et des flèches
for nom, (conditions, conclusions) in regles_td.items():
    for fait in conditions:
        graphe.add_edge(fait, nom)           # flèche : condition → règle
    for fait in conclusions:
        graphe.add_edge(nom, fait)           # flèche : règle → conclusion
print(graphe.number_of_nodes(), "sommets et", graphe.number_of_edges(), "flèches")
sans_fleche_entrante = [n for n in graphe if graphe.in_degree(n) == 0]
print("Faits sur lesquels n'arrive aucune flèche :", sorted(sans_fleche_entrante))

### Ce que vous devez voir
```
16 sommets et 21 flèches
Faits sur lesquels n'arrive aucune flèche : ['G', 'H', 'K']
```
16 sommets (9 faits et 7 règles) et 21 flèches : 13 conditions et 8 conclusions. Aucune flèche n'arrive sur G, H et K : ce sont bien les faits non déductibles de l'énoncé.

**Avant d'exécuter la cellule suivante :** sur votre brouillon, tracez le chemin qui mène de H et K jusqu'à C. Passe-t-il par R4 ?

In [ ]:
#@title Dessiner le graphe : rectangles = faits, cercles = règles
import matplotlib.pyplot as plt

position = {"H": (2, 0), "R3": (2, 1), "F": (1, 1.5), "G": (0, 2.5), "K": (0, 4.5),
            "R7": (1, 3), "A": (2, 3), "R1": (3, 3), "E": (4, 3), "R4": (5, 1.5),
            "R5": (5, 4.5), "B": (6, 4.5), "R2": (7, 4.5), "D": (8, 4.5), "R6": (9, 3), "C": (10, 3)}
les_regles = [n for n in graphe if n.startswith("R")]
les_faits = [n for n in graphe if not n.startswith("R")]
plt.figure(figsize=(10, 5))
nx.draw_networkx_nodes(graphe, position, nodelist=les_faits, node_shape="s", node_size=800, node_color="#dbe9f6", edgecolors="black")
nx.draw_networkx_nodes(graphe, position, nodelist=les_regles, node_shape="o", node_size=800, node_color="#fde2c4", edgecolors="black")
nx.draw_networkx_labels(graphe, position)
nx.draw_networkx_edges(graphe, position, node_size=800, arrowsize=15)
plt.title("Graphe ET/OU des règles R1 à R7 (TD du chapitre 3, exercice 1)")
plt.axis("off")
plt.show()

### Ce que vous devez voir
Un graphe de 16 sommets : H en bas, sous R3 ; K et G à gauche ; C à droite, au bout de la chaîne R1 → E → R5 → B → R2 → D → R6. Les flèches qui entrent dans une même règle sont liées par ET ; quand deux règles aboutissent au même fait (R3 et R7 pour A, R4 et R6 pour C), elles sont liées par OU.

**Observez :** G n'est ni dans la base initiale ni déductible. Les règles R7 et R4, qui en ont besoin, ne pourront donc jamais se déclencher, et le seul chemin possible vers C passe par R6. C'est le diagnostic du corrigé.

### b) Dérouler le chaînage avant
Le corrigé du TD examine les règles dans l'ordre imposé. Une règle se déclenche si toutes ses conditions sont dans la base de faits ; ses conclusions sont alors ajoutées à la base. Une règle déjà déclenchée n'est pas rejouée. On s'arrête dès que le but est dans la base, ou quand un **passage** complet sur les règles n'ajoute plus rien.

La cellule « À MODIFIER » prépare l'ordre des règles, la base de faits initiale et le but. Tapez les noms séparés par des espaces.

**Avant d'exécuter :** que vont afficher les deux dernières lignes ?

In [ ]:
#@title À MODIFIER : ordre des règles, base de faits initiale et but
ordre = "R3 R7 R1 R4 R5 R2 R6"  #@param {type:"string"}
base_initiale = "H K"  #@param {type:"string"}
but = "C"  #@param ["C", "D", "B", "E", "A"]

regles_ordonnees = []
for nom in ordre.upper().split():           # "R3 R7 ..." devient ["R3", "R7", ...]
    if nom in regles_td:                    # on ignore les noms inconnus
        regles_ordonnees.append((nom, regles_td[nom]))
faits_initiaux = set(base_initiale.upper().split())
print("Ordre :", [nom for nom, regle in regles_ordonnees])
print("Base de faits initiale :", sorted(faits_initiaux), "| but :", but)

### Ce que vous devez voir
```
Ordre : ['R3', 'R7', 'R1', 'R4', 'R5', 'R2', 'R6']
Base de faits initiale : ['H', 'K'] | but : C
```
**Avant d'exécuter la cellule suivante :** dans quel ordre les règles vont-elles se déclencher ? Le but C sera-t-il atteint, et en combien de passages ?

La cellule suivante reprend le programme du corrigé du TD en numérotant les passages. Il y a au plus un passage par règle, puisque chaque passage utile déclenche au moins une règle nouvelle.

In [ ]:
#@title b) Le chaînage avant du TD, passage par passage
faits = set(faits_initiaux)                   # copie de la base de départ
declenchees = []
for passage in range(1, len(regles_ordonnees) + 1):   # un passage = un tour complet des règles
    nouveau = False
    for nom, (conditions, conclusions) in regles_ordonnees:
        # une règle se déclenche une seule fois, si ses conditions sont dans la base (<=)
        if nom not in declenchees and conditions <= faits and but not in faits:
            faits = faits | conclusions       # ajouter les conclusions
            declenchees.append(nom)
            nouveau = True
            print(f"Passage {passage} · {nom} déclenchée -> {sorted(faits)}")
    if not nouveau or but in faits:           # rien de nouveau, ou but atteint : arrêt
        break
print("But atteint :", but in faits, "| règles déclenchées :", declenchees)

### Ce que vous devez voir
```
Passage 1 · R3 déclenchée -> ['A', 'F', 'H', 'K']
Passage 1 · R1 déclenchée -> ['A', 'E', 'F', 'H', 'K']
Passage 1 · R5 déclenchée -> ['A', 'B', 'E', 'F', 'H', 'K']
Passage 1 · R2 déclenchée -> ['A', 'B', 'D', 'E', 'F', 'H', 'K']
Passage 1 · R6 déclenchée -> ['A', 'B', 'C', 'D', 'E', 'F', 'H', 'K']
But atteint : True | règles déclenchées : ['R3', 'R1', 'R5', 'R2', 'R6']
```
Le but C est établi dès le premier passage. La base de faits finale est {H, K, A, F, E, B, D, C} et les règles déclenchées sont, dans l'ordre, **R3, R1, R5, R2, R6**. R7 et R4 ne se déclenchent jamais, puisque G n'est ni dans la base initiale ni déductible.

**Modifiez**, dans la cellule « À MODIFIER », puis relancez les deux cellules :
1. `base_initiale` = `H G K` : R7 et R4 peuvent alors se déclencher, et le but est atteint plus tôt, par R4. Quelles règles se déclenchent ?
2. `ordre` = `R6 R2 R5 R4 R1 R7 R3` (l'ordre inverse) : les mêmes règles se déclenchent-elles ? Combien de passages faut-il maintenant ?
3. `base_initiale` = `K` : pourquoi rien ne se déclenche-t-il ?

### Le tableau du corrigé, étape par étape
Le corrigé présente le premier passage sous forme de tableau : chaque règle examinée, sa condition, si elle se déclenche et la base de faits après l'examen. La bibliothèque `pandas` range ces lignes dans un tableau (`DataFrame`).

**Avant d'exécuter :** combien de lignes compte le tableau ? Que dit la colonne « Déclenchée ? » pour R7 et pour R4 ?

In [ ]:
#@title Le tableau du premier passage, comme dans le corrigé
import pandas as pd

faits = set(faits_initiaux)
lignes = []
for etape, (nom, (conditions, conclusions)) in enumerate(regles_ordonnees, start=1):
    if conditions <= faits:
        faits = faits | conclusions
        reponse = "Oui"
    else:
        reponse = "Non : " + ", ".join(sorted(conditions - faits)) + " absent"
    lignes.append([etape, nom, " ∧ ".join(sorted(conditions)), reponse, ", ".join(sorted(faits))])
    if but in faits:
        break
pd.DataFrame(lignes, columns=["Étape", "Règle examinée", "Condition", "Déclenchée ?", "Base de faits après"])

### Ce que vous devez voir
| | Étape | Règle examinée | Condition | Déclenchée ? | Base de faits après |
|---|---|---|---|---|---|
| 0 | 1 | R3 | H | Oui | A, F, H, K |
| 1 | 2 | R7 | F ∧ G ∧ K | Non : G absent | A, F, H, K |
| 2 | 3 | R1 | A | Oui | A, E, F, H, K |
| 3 | 4 | R4 | E ∧ G | Non : G absent | A, E, F, H, K |
| 4 | 5 | R5 | E ∧ K | Oui | A, B, E, F, H, K |
| 5 | 6 | R2 | B | Oui | A, B, D, E, F, H, K |
| 6 | 7 | R6 | D ∧ E ∧ K | Oui | A, B, C, D, E, F, H, K |

La colonne de gauche, sans titre, est l'index du tableau (0 à 6).

C'est le tableau du corrigé, à l'ordre d'écriture des faits près : ici, la base est triée par ordre alphabétique. Comparez-le avec le vôtre, ligne par ligne.

**Observez :** avec `base_initiale` = `H G K`, le tableau s'arrête à l'étape 4, sur R4. Le moteur a-t-il « compris » quelque chose ? Justifiez votre réponse : fait-il un raisonnement, ou vérifie-t-il simplement qu'un ensemble est inclus dans un autre ?

## À vous
1. À l'étape 4, décrivez le sonnet 18 de Shakespeare (3 quatrains et 1 distique), puis un haïku (3 vers, 5-7-5 syllabes) : vérifiez que les règles déclenchées sont celles que vous aviez prévues.
2. Dans l'exercice du TD, ajoutez G à la base initiale, puis changez l'ordre des règles : notez ce qui change et ce qui ne change pas.
3. Le système ignore la villanelle, forme fixe de 19 vers : 5 tercets et 1 quatrain, par exemple Jean Passerat, « J'ai perdu ma tourterelle » (1574). La cellule ci-dessous ajoute une règle R8 écrite dans les champs, puis lance le moteur sur le poème que vous décrivez. Séparez les faits par des virgules.

In [ ]:
#@title À MODIFIER : ajoutez votre propre règle, puis décrivez un poème
si = "19 vers, 5 tercets, 1 quatrain"  #@param {type:"string"}
alors = "villanelle"  #@param {type:"string"}
faits_du_poeme = "19 vers, 5 tercets, 1 quatrain"  #@param {type:"string"}

premisses = set()
for morceau in si.split(","):              # "19 vers, 5 tercets" → {"19 vers", "5 tercets"}
    premisses.add(morceau.strip())         # strip() retire les espaces en trop
faits = set()
for morceau in faits_du_poeme.split(","):
    faits.add(morceau.strip())
regles_etendues = regles + [(premisses, alors.strip())]   # R8 : la nouvelle règle, en dernier
resultat = chainage_avant(faits, regles_etendues)
print("Faits déduits :", sorted(resultat - faits))

### Ce que vous devez voir
```
Cycle 1 : R8 déclenchée → villanelle
Cycle 2 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['villanelle']
```
**Observez :** « forme fixe » n'a pas été déduit, alors que la villanelle en est une. Que faudrait-il ajouter à la base de règles ? Chaque nouvelle forme demande plusieurs règles, écrites à la main : c'est le **goulot d'étranglement de l'acquisition des connaissances**. Essayez ensuite `faits_du_poeme` = `19 vers, 5 tercets, 1 quatrain, sonnet` : qu'arrive-t-il, et pourquoi le système ne voit-il pas la contradiction ?

## Pour le mini-projet : votre propre base de règles
Le moteur ne dépend pas du domaine : vous pouvez écrire vos règles dans un tableur, une règle par ligne, avec trois colonnes `regle`, `si` (les prémisses, séparées par des virgules) et `alors` (la conclusion). Enregistrez-le au format « CSV UTF-8 » avec des **points-virgules** comme séparateurs, puisque les prémisses contiennent des virgules. Les règles de l'exemple, qui reconnaissent le genre d'un texte, sont **simplifiées et fictives** : à discuter avant tout usage.

- Dans Colab, cochez `charger_mon_fichier` : une fenêtre s'ouvre pour choisir le fichier.
- Dans Jupyter, placez le fichier à côté du carnet et écrivez son nom dans `nom_du_fichier`.

**Avant d'exécuter :** combien de règles et de colonnes le tableau d'exemple compte-t-il ?

In [ ]:
#@title Une base de règles dans un tableau (exemple fictif : le genre d'un texte)
import io

exemple_regles = """regle;si;alors
R1;répliques, didascalies;texte de théâtre
R2;vers, rimes;poème versifié
R3;narrateur, chapitres;récit
R4;récit, long;roman
R5;récit, court;nouvelle"""
tableau_regles = pd.read_csv(io.StringIO(exemple_regles), sep=";")   # sep : le séparateur
print(tableau_regles.shape)
tableau_regles

### Ce que vous devez voir
`(5, 3)`, puis un tableau de 5 règles et 3 colonnes : `regle`, `si`, `alors`. R3 et R4 s'enchaînent : la conclusion de R3 (« récit ») est une prémisse de R4 et de R5.

In [ ]:
#@title À MODIFIER : charger votre propre base de règles (facultatif)
charger_mon_fichier = False  #@param {type:"boolean"}
nom_du_fichier = "mes_regles.csv"  #@param {type:"string"}

if charger_mon_fichier:
    try:
        from google.colab import files        # ce module n'existe que dans Colab
        envoi = files.upload()                # une fenêtre s'ouvre : choisissez votre fichier
        nom_du_fichier = list(envoi)[0]
    except ImportError:
        print("Hors de Colab : le fichier", nom_du_fichier, "doit être à côté du carnet.")
    tableau_regles = pd.read_csv(nom_du_fichier, sep=";", encoding="utf-8")
print(len(tableau_regles), "règles")
print(tableau_regles.head())

### Ce que vous devez voir
Tant que la case n'est pas cochée, les 5 règles de l'exemple. Avec votre fichier, vérifiez que les trois colonnes s'appellent bien `regle`, `si` et `alors`.

**Avant d'exécuter la cellule suivante :** pour un texte qui a un narrateur, des chapitres et qui est long, quelles règles vont se déclencher ?

In [ ]:
#@title À MODIFIER : faire raisonner le moteur sur votre base de règles
faits_du_texte = "narrateur, chapitres, long"  #@param {type:"string"}

regles_du_tableau = []
for i, ligne in tableau_regles.iterrows():    # une ligne du tableau = une règle
    premisses = set()
    for morceau in ligne["si"].split(","):
        premisses.add(morceau.strip())
    regles_du_tableau.append((premisses, ligne["alors"].strip()))
faits = set()
for morceau in faits_du_texte.split(","):
    faits.add(morceau.strip())
resultat = chainage_avant(faits, regles_du_tableau)   # R1 = première ligne du tableau
print("Faits déduits :", sorted(resultat - faits))

### Ce que vous devez voir
```
Cycle 1 : R3 déclenchée → récit
Cycle 2 : R4 déclenchée → roman
Cycle 3 : aucune règle ne produit de fait nouveau → arrêt
Faits déduits : ['roman', 'récit']
```
R3 conclut « récit », ce qui permet à R4 de conclure « roman ». Le même moteur, sans aucune modification, raisonne maintenant sur les genres.

**Pour votre mini-projet :** écrivez une dizaine de règles de votre spécialité (reconnaître un registre de langue, une figure de style, le niveau d'un texte pour des apprenants…), testez-les sur des cas que vous connaissez, puis cherchez un cas limite qui les met en défaut.

## À rendre
Répondez par écrit, en quelques lignes chacune :
1. Expliquez avec vos propres mots la différence entre chaînage avant et chaînage arrière, en vous appuyant sur les deux traces obtenues pour le sonnet de Du Bellay (étapes 4 et 6).
2. Le sonnet imprimé d'un seul bloc ne déclenche aucune règle, alors qu'un lecteur le reconnaît aussitôt. Que révèle cet échec sur les forces et les limites des systèmes experts ?
3. Recopiez le déroulement obtenu pour l'exercice 1 du TD avec la base {H, K}, puis indiquez ce qui change quand on ajoute G à la base initiale. Pourquoi le but est-il atteint plus tôt ?

## Ce qu'il faut retenir
- Un système expert réunit une base de faits, une base de règles SI … ALORS et un moteur d'inférence ; le moteur ne dépend pas du domaine.
- Le chaînage avant va des faits vers le but, cycle après cycle, en suivant l'ordre de priorité des règles ; le chaînage arrière va du but vers les faits, par sous-buts.
- Sa force est l'explicabilité : chaque conclusion se justifie par la chaîne des règles déclenchées. Ses limites : il faut écrire toutes les règles à la main, et il échoue hors des cas prévus.

**Dans le cours :** [IA symbolique : les systèmes experts](https://progremer04.github.io/canva-au-for-ang-/#c3-symbolique) · [TD du chapitre 3, exercice 1](https://progremer04.github.io/canva-au-for-ang-/#td-ch3-ex1)